# Phase 5 -- Exploratory Data Analysis

This notebook is a **narrative wrapper**, not a second implementation: every
analysis and chart below is a thin call into `creditguard.eda` (`univariate.py`,
`bivariate.py`, `risk_analysis.py`, `plots.py`), the same functions
`python -m creditguard.eda.run_eda` uses to regenerate every figure headlessly
for CI. See [`docs/feature_dictionary.md`](../docs/feature_dictionary.md) for
what each feature means and [`reports/eda/findings.md`](../reports/eda/findings.md)
for the write-up this notebook's evidence feeds into.

**Two jobs, one notebook:** produce the portfolio charts the Phase 9 dashboard
will reuse, and produce evidence for Phase 6 modelling decisions (class
imbalance handling, log-transform candidates, multicollinearity, leakage
re-checks, and whether the Phase 4 temporal split crosses a regime shift).

**Where each analysis runs (train split vs. full population) is deliberate** --
see the module docstring in `src/creditguard/eda/run_eda.py` for the full
reasoning. Short version: portfolio-level charts (distributions, frequencies,
band default rates, the temporal check) use the **full** train+val+test
population; IV/WOE, the correlation matrix and point-biserial correlations --
anything meant to inform what Phase 6 actually fits -- use the **train split
only**, the same fold the model will train on.

In [1]:
import os
from pathlib import Path

if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
print("Working directory:", Path.cwd())

Working directory: C:\Users\asiff\OneDrive\Documents\CreditGuard


In [2]:
import warnings

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from creditguard.eda import bivariate, plots, risk_analysis, univariate
from creditguard.eda.run_eda import (
    BAND_BREAKDOWNS,
    CATEGORICAL_FREQUENCY_COLUMNS,
    UNIVARIATE_NUMERIC_COLUMNS,
    build_eda_frame,
    load_features_config,
)
from creditguard.validation.engine import load_rule_config

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

DATASET_VERSION = "ds_20260808_547ecf5a_clean"
TARGET_COL = "default_12m"

features_config = load_features_config("config/features.yaml")
cleaning_config = load_rule_config("config/validation_rules.yaml")
numeric_columns = features_config["feature_columns"]["numeric"]
categorical_columns = features_config["feature_columns"]["categorical"]
ordinal_columns = features_config["feature_columns"]["ordinal"]
print(f"{len(numeric_columns)} numeric, {len(categorical_columns)} categorical, "
      f"{len(ordinal_columns)} ordinal feature columns configured.")

43 numeric, 6 categorical, 4 ordinal feature columns configured.


In [3]:
full = build_eda_frame(DATASET_VERSION, features_config, cleaning_config)
train = full[full["split"] == "train"].reset_index(drop=True)

print(f"Full population: {len(full):,} loans")
for split_name in ("train", "val", "test"):
    subset = full[full["split"] == split_name]
    print(f"  {split_name}: {len(subset):,} loans, "
          f"{subset['application_date'].min().date()} .. {subset['application_date'].max().date()}")

Full population: 96,749 loans
  train: 67,724 loans, 2022-08-08 .. 2024-09-14
  val: 14,513 loans, 2024-09-15 .. 2025-02-25
  test: 14,512 loans, 2025-02-25 .. 2025-08-08


## 1. Class balance and univariate distributions

Full population (portfolio-level view).

In [4]:
rate_summary = univariate.default_rate_summary(full[TARGET_COL])
print(rate_summary)
fig = plots.plot_class_balance(full[TARGET_COL])
display(fig)
plt.close(fig)

{'n': 96749, 'n_default': 10738, 'n_non_default': 86011, 'default_rate': 0.1109882272684989}


<Figure size 500x400 with 1 Axes>

In [5]:
numeric_summary_table = univariate.numeric_summary(full, list(UNIVARIATE_NUMERIC_COLUMNS))
display(numeric_summary_table)

,n,n_missing,mean,std,min,p25,median,p75,max,skew,kurtosis,log_transform_flag
feature,,,,,,,,,,,,
age,96749,0,39.4600,10.1562,18.0000,32.0000,39.0000,46.0000,100.0000,0.7883,2.4620,False
annual_income,96749,0,"409,398.9202","256,424.1901","62,305.8104","225,591.5508","346,389.7601","525,808.4413","1,338,453.5709",1.3358,1.8494,True
monthly_income,96749,0,"34,095.2280","21,394.3873","5,428.5919","18,743.5017","28,824.6249","43,804.2047","111,022.7670",1.3320,1.8179,True
loan_amount,96749,0,"480,915.0093","670,317.5391","22,537.5939","117,446.7334","234,642.3695","512,082.4493","3,810,880.7963",2.8857,9.1385,True
interest_rate,96749,0,13.1076,5.4943,2.0116,9.3646,11.9383,15.0848,32.4137,1.0052,0.4797,True
loan_tenure_months,96749,0,70.9680,78.7306,6.0000,25.0000,43.0000,71.0000,360.0000,2.1104,3.5804,True
dti,96749,0,0.6810,0.1575,0.2898,0.5654,0.6752,0.7889,1.4003,0.2399,-0.2489,False
post_loan_dti,96749,0,0.9995,0.2846,0.3896,0.8088,0.9598,1.1381,5.0000,1.5620,6.3533,True
credit_utilization,96749,0,0.3416,0.1650,0.0118,0.2145,0.3223,0.4480,1.0538,0.5546,-0.0992,False


In [6]:
for column in UNIVARIATE_NUMERIC_COLUMNS:
    fig = plots.plot_numeric_distribution(full, column)
    display(fig)
    plt.close(fig)

<Figure size 600x500 with 2 Axes>

<Figure size 600x500 with 2 Axes>

<Figure size 600x500 with 2 Axes>

<Figure size 600x500 with 2 Axes>

<Figure size 600x500 with 2 Axes>

...

<Figure size 600x500 with 2 Axes>

<Figure size 600x500 with 2 Axes>

<Figure size 600x500 with 2 Axes>

<Figure size 600x500 with 2 Axes>

<Figure size 600x500 with 2 Axes>

## 2. Categorical frequency tables

In [7]:
freq_tables = univariate.categorical_frequency(full, list(CATEGORICAL_FREQUENCY_COLUMNS))
for column, table in freq_tables.items():
    print(f"\n{column}")
    display(table)
    fig = plots.plot_categorical_frequency(full, column)
    display(fig)
    plt.close(fig)


gender


,count,pct
gender,,
FEMALE,47766,49.3700
MALE,47031,48.6100
OTHER,1952,2.0200


<Figure size 600x250 with 1 Axes>


marital_status


,count,pct
marital_status,,
MARRIED,53183,54.9700
SINGLE,33945,35.0900
DIVORCED,6819,7.0500
WIDOWED,2802,2.9000


...

,count,pct
loan_purpose,,
DEBT_CONSOLIDATION,17397,17.9800
HOME_IMPROVEMENT,11683,12.0800
MEDICAL,11668,12.0600
EDUCATION,11530,11.9200
OTHER,9728,10.0500
BUSINESS_EXPANSION,9714,10.0400
VEHICLE_PURCHASE,9652,9.9800
TRAVEL,7766,8.0300
WEDDING,7611,7.8700


<Figure size 600x450 with 1 Axes>


city_tier


,count,pct
city_tier,,
2,38744,40.0500
1,33899,35.0400
3,24106,24.9200


<Figure size 600x250 with 1 Axes>

## 3. Default rate by decile -- "the single most useful chart in credit risk"

A curated set of key numeric drivers, shown here for narrative purposes;
`run_eda` generates this chart for **every** numeric feature (`n_features`
figures under `reports/figures/eda/decile_*.png`).

In [8]:
KEY_DRIVERS = (
    "dti",
    "credit_utilization",
    "loan_to_income",
    "savings_to_income",
    "credit_history_years",
    "age",
    "annual_income",
    "previous_defaults",
)
for column in KEY_DRIVERS:
    decile_df = bivariate.default_rate_by_decile(full, column, TARGET_COL)
    display(decile_df)
    fig = plots.plot_default_rate_by_decile(decile_df)
    display(fig)
    plt.close(fig)

,feature,decile,bin,min_value,max_value,n,n_default,default_rate
0,dti,1,"(0.289, 0.477]",0.2898,0.4766,9675,508,0.0525
1,dti,2,"(0.477, 0.54]",0.4766,0.5403,9675,658,0.0680
2,dti,3,"(0.54, 0.59]",0.5403,0.5899,9675,728,0.0752
3,dti,4,"(0.59, 0.634]",0.5899,0.6338,9675,862,0.0891
4,dti,5,"(0.634, 0.675]",0.6338,0.6752,9675,929,0.0960
5,dti,6,"(0.675, 0.718]",0.6752,0.7183,9674,1000,0.1034
6,dti,7,"(0.718, 0.764]",0.7183,0.7639,9675,1152,0.1191
7,dti,8,"(0.764, 0.817]",0.7639,0.8174,9675,1309,0.1353
8,dti,9,"(0.817, 0.889]",0.8174,0.8887,9675,1553,0.1605
9,dti,10,"(0.889, 1.4]",0.8887,1.4003,9675,2039,0.2107


<Figure size 700x450 with 2 Axes>

,feature,decile,bin,min_value,max_value,n,n_default,default_rate
0,credit_utilization,1,"(0.0108, 0.142]",0.0118,0.1416,9675,431,0.0445
1,credit_utilization,2,"(0.142, 0.192]",0.1416,0.1915,9675,725,0.0749
2,credit_utilization,3,"(0.192, 0.236]",0.1915,0.2361,9675,785,0.0811
3,credit_utilization,4,"(0.236, 0.278]",0.2361,0.2784,9675,874,0.0903
4,credit_utilization,5,"(0.278, 0.322]",0.2784,0.3223,9675,978,0.1011
5,credit_utilization,6,"(0.322, 0.368]",0.3223,0.3683,9674,1044,0.1079
6,credit_utilization,7,"(0.368, 0.421]",0.3683,0.4211,9675,1108,0.1145
7,credit_utilization,8,"(0.421, 0.481]",0.4211,0.4812,9676,1239,0.1280
8,credit_utilization,9,"(0.481, 0.571]",0.4813,0.5706,9674,1488,0.1538
9,credit_utilization,10,"(0.571, 1.054]",0.5706,1.0538,9675,2066,0.2135


<Figure size 700x450 with 2 Axes>

,feature,decile,bin,min_value,max_value,n,n_default,default_rate
0,loan_to_income,1,"(0.0608, 0.232]",0.0618,0.2315,9675,504,0.0521
1,loan_to_income,2,"(0.232, 0.372]",0.2316,0.3717,9675,572,0.0591
2,loan_to_income,3,"(0.372, 0.469]",0.3717,0.4689,9675,643,0.0665
3,loan_to_income,4,"(0.469, 0.554]",0.4689,0.5538,9675,664,0.0686
4,loan_to_income,5,"(0.554, 0.647]",0.5538,0.6473,9675,694,0.0717
5,loan_to_income,6,"(0.647, 0.772]",0.6473,0.7723,9674,710,0.0734
6,loan_to_income,7,"(0.772, 0.98]",0.7723,0.9799,9675,766,0.0792
7,loan_to_income,8,"(0.98, 1.532]",0.9799,1.5315,9675,915,0.0946
8,loan_to_income,9,"(1.532, 3.462]",1.5315,3.4618,9675,1551,0.1603
9,loan_to_income,10,"(3.462, 10.0]",3.4618,10.0000,9675,3719,0.3844


...

<Figure size 700x450 with 2 Axes>

,feature,decile,bin,min_value,max_value,n,n_default,default_rate
0,annual_income,1,"(62305.809, 149102.081]","62,305.8104","149,096.6091",9675,1991,0.2058
1,annual_income,2,"(149102.081, 202266.539]","149,103.4495","202,264.5111",9675,1572,0.1625
2,annual_income,3,"(202266.539, 248281.223]","202,267.8915","248,279.9502",9675,1239,0.1281
3,annual_income,4,"(248281.223, 296013.672]","248,283.1327","296,010.4828",9675,1212,0.1253
4,annual_income,5,"(296013.672, 346389.76]","296,026.4285","346,389.7601",9675,1019,0.1053
5,annual_income,6,"(346389.76, 405195.347]","346,393.1562","405,149.7122",9674,968,0.1001
6,annual_income,7,"(405195.347, 478088.411]","405,206.7562","478,088.4110",9676,858,0.0887
7,annual_income,8,"(478088.411, 581830.471]","478,117.0059","581,830.4709",9675,758,0.0783
8,annual_income,9,"(581830.471, 760062.516]","581,850.2336","760,062.5163",9675,629,0.0650
9,annual_income,10,"(760062.516, 1338453.571]","760,110.5502","1,338,453.5709",9674,492,0.0509


<Figure size 700x450 with 2 Axes>

,feature,decile,bin,min_value,max_value,n,n_default,default_rate
0,previous_defaults,1,"(-0.001, 10.0]",0.0000,10.0000,96749,10738,0.1110


<Figure size 700x450 with 2 Axes>

## 4. Default rate by band

Age band, income band, employment type, loan type, credit history band,
utilization band, dependents, city tier -- the full set the Phase 5 brief
asks for.

In [9]:
for band_col, order in BAND_BREAKDOWNS:
    band_df = bivariate.default_rate_by_band(full, band_col, TARGET_COL, order)
    display(band_df)
    fig = plots.plot_default_rate_by_band(band_df, band_col)
    display(fig)
    plt.close(fig)

,age_band,n,n_default,default_rate
0,Q1,22238,4276,0.1923
1,Q2,17487,2419,0.1383
2,Q3,18596,1854,0.0997
3,Q4,20842,1467,0.0704
4,Q5,17586,722,0.0411


<Figure size 500x450 with 1 Axes>

,income_band,n,n_default,default_rate
0,Q1,19359,3566,0.1842
1,Q2,19377,2451,0.1265
2,Q3,19488,2006,0.1029
3,Q4,19254,1598,0.0830
4,Q5,19271,1117,0.0580


<Figure size 500x450 with 1 Axes>

,employment_type,n,n_default,default_rate
0,BUSINESS_OWNER,11983,1156,0.0965
1,SALARIED,60474,6478,0.1071
2,SELF_EMPLOYED,21061,2389,0.1134
3,UNEMPLOYED,3231,715,0.2213


...

<Figure size 500x450 with 1 Axes>

,dependents,n,n_default,default_rate
0,0,32251,3388,0.1051
1,1,35467,3970,0.1119
2,2,19344,2259,0.1168
3,3,7177,814,0.1134
4,4,2000,229,0.1145
5,5,416,59,0.1418
6,6,94,19,0.2021


<Figure size 630x450 with 1 Axes>

,city_tier,n,n_default,default_rate
0,1,33899,3401,0.1003
1,2,38744,4329,0.1117
2,3,24106,3008,0.1248


<Figure size 500x450 with 1 Axes>

## 5. Information Value / Weight of Evidence (train split only)

IV bands: `<0.02` useless, `0.02-0.1` weak, `0.1-0.3` medium, `0.3-0.5`
strong, `>0.5` suspiciously strong -- investigate for leakage.

In [10]:
iv_df = risk_analysis.iv_table(
    train, numeric_columns, list(categorical_columns) + list(ordinal_columns), TARGET_COL
)
display(iv_df)
fig = plots.plot_iv_table(iv_df)
display(fig)
plt.close(fig)

,feature,type,iv,interpretation
0,loan_to_income,numeric,0.6596,suspiciously strong - investigate for leakage
1,loan_type,categorical,0.6010,suspiciously strong - investigate for leakage
2,loan_tenure_months,numeric,0.4869,strong
3,post_loan_dti,numeric,0.4787,strong
4,savings_balance,numeric,0.4756,strong
5,credit_history_years,numeric,0.3769,strong
6,credit_history_months,numeric,0.3769,strong
7,months_of_runway,numeric,0.3739,strong
8,disposable_income,numeric,0.3715,strong
9,age,numeric,0.3479,strong


<Figure size 700x1590 with 1 Axes>

## 6. Correlation structure and multicollinearity (train split only)

In [11]:
corr = bivariate.correlation_matrix(train, numeric_columns)
fig = plots.plot_correlation_heatmap(corr)
display(fig)
plt.close(fig)

high_corr = bivariate.high_correlation_pairs(corr, threshold=0.8)
print(f"{len(high_corr)} pair(s) with |r| > 0.8:")
for pair in high_corr:
    print(f"  {pair['feature_a']} <-> {pair['feature_b']}: r={pair['r']:.3f}")

<Figure size 1720x1720 with 2 Axes>

20 pair(s) with |r| > 0.8:
  credit_history_months <-> credit_history_years: r=1.000
  existing_loan_amount <-> monthly_emi: r=0.996
  annual_income <-> monthly_income: r=0.995
  total_liabilities <-> total_outstanding: r=0.992
  annual_income <-> monthly_expenses: r=0.933
  monthly_income <-> monthly_expenses: r=0.930
  age <-> credit_history_years: r=0.919
  age <-> credit_history_months: r=0.919
  savings_to_income <-> months_of_runway: r=0.909
  previous_defaults <-> has_prior_default: r=0.840
  annual_income <-> total_credit_limit: r=0.836
  employment_years <-> credit_history_years: r=0.835
  employment_years <-> credit_history_months: r=0.835
  monthly_income <-> total_credit_limit: r=0.832
  loan_tenure_months <-> loan_to_income: r=0.823
  age <-> employment_years: r=0.813
  total_liabilities <-> total_credit_limit: r=0.806
  closed_loans <-> credit_history_years: r=0.803
  credit_history_months <-> closed_loans: r=0.803
  existing_loan_count <-> active_loans: r=0.802


## 7. Point-biserial correlation with the target (train split only)

In [12]:
point_biserial = bivariate.point_biserial_correlations(train, numeric_columns, TARGET_COL)
display(point_biserial)

,feature,point_biserial_r,p_value
0,loan_to_income,0.3264,0.0000
1,loan_tenure_months,0.2598,0.0000
2,post_loan_dti,0.2007,0.0000
3,credit_history_years,-0.1806,0.0000
4,credit_history_months,-0.1806,0.0000
5,loan_amount,0.1790,0.0000
6,employment_years,-0.1767,0.0000
7,months_of_runway,-0.1732,0.0000
8,savings_balance,-0.1706,0.0000
9,age,-0.1682,0.0000


## 8. Temporal check -- does the Phase 4 split cross a regime shift?

Full population, by calendar month of `application_date`.

In [13]:
monthly = risk_analysis.monthly_volume_and_default_rate(full)
display(monthly)
fig = plots.plot_temporal_trend(monthly)
display(fig)
plt.close(fig)

regime = risk_analysis.detect_regime_shift(monthly)
print(regime)

,month,n,n_default,default_rate
0,2022-08,2064,255,0.1235
1,2022-09,2564,320,0.1248
2,2022-10,2743,330,0.1203
3,2022-11,2673,320,0.1197
4,2022-12,2747,314,0.1143
5,2023-01,2816,348,0.1236
6,2023-02,2532,315,0.1244
7,2023-03,2699,328,0.1215
8,2023-04,2677,294,0.1098
9,2023-05,2712,286,0.1055


<Figure size 900x450 with 2 Axes>

{'mean_default_rate': 0.11121324065504894, 'std_default_rate': 0.007615025779535768, 'flagged_months': ['2024-09'], 'regime_shift_detected': True}


## Regenerating everything headlessly

```bash
python -m creditguard.eda.run_eda --dataset-version ds_20260808_547ecf5a_clean
```

regenerates every figure under `reports/figures/eda/` (150-dpi PNG) and every
summary table under `reports/eda/tables/`, headlessly, so CI can run it. The
evidenced findings and the **Decisions for Phase 6** this EDA feeds into are
written up in [`reports/eda/findings.md`](../reports/eda/findings.md).